# 04. Digital Cell-Type Sorting

Assigns individual long-read sequencing fragments to one of seven blood cell populations
using log-likelihood ratio (LLR) scoring against purified-cell marker atlases.

**Pipeline overview:**
1. Download the seven target/background marker BED files produced by notebook 03
2. Build per-chromosome methylation probability arrays for all 7 targets
3. For each sample: stream the PAT file, score every read against all 7 atlases simultaneously
4. Write reads with `LLR > log(τ)` (τ=1.053) and `hits ≥ min_hits` (5 CpGs) to per-target PATs
5. bgzip + `wgbstools index` + `pat2beta` each target output
6. Upload to `results/cell_sorted/{CellType}/`

**Targets:** Myeloid, Lymphoid, T_Cell, B_Cell, NK_Cell, Monocyte, Granulocyte

**Mode:** `MODE = "TEST"` processes 3 samples end-to-end;
`MODE = "PRODUCTION"` processes all batches sequentially.
For large-scale parallel runs use **04b_dsub_sorting.ipynb** instead.

## Cell 1 — Imports

In [ ]:
import os
import re
import math
import gzip
import shutil
import subprocess
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

print("Python environment ready.")

## Cell 2 — Configuration

Set `MODE = "TEST"` to validate the pipeline on 3 samples, or `"PRODUCTION"` for the full run.
Set `ACTIVE_BATCH` to a single GCS prefix to process one batch at a time, or `None` for all.

In [ ]:
WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]
HOME = Path(os.getcwd())

# ── Mode ─────────────────────────────────────────────────────────
MODE = "TEST"   # "TEST" (3 samples) | "PRODUCTION" (all, skip_if_exists=True)

TEST_SAMPLE_LIMIT = 3      # samples to process in TEST mode
MIN_HITS          = 5      # minimum informative CpGs per read
TAU               = 1.053  # LLR threshold; log(tau) ≈ 0.052 nats
THREADS           = 4      # pat2beta threads
MAKE_BETA         = True   # produce .beta in addition to .pat.gz / .csi

# ── wgbstools ────────────────────────────────────────────────────
TOOL_BIN        = HOME / "wgbs_tools" / "wgbstools"
CPG_CHROME_SIZE = HOME / "wgbs_tools" / "references" / "hg38" / "CpG.chrome.size"

# ── Local scratch ─────────────────────────────────────────────────
TMP           = HOME / "tmp_cell_sorting"
MARKERS_LOCAL = HOME / "markers_all7"
TMP.mkdir(parents=True, exist_ok=True)
MARKERS_LOCAL.mkdir(parents=True, exist_ok=True)

# ── GCS output root ───────────────────────────────────────────────
# Files land at: results/cell_sorted/{CellType}/{sample_key}_{CellType}.*
OUT_ROOT_GCS = f"{WORKSPACE_BUCKET}/results/cell_sorted"

# ── Marker BED paths (produced by notebook 03) ───────────────────
MARKERS_BASE = f"{WORKSPACE_BUCKET}/results/markers/markers_S2"
TARGET_BEDS = {
    "Myeloid":     f"{MARKERS_BASE}/coarse_Myeloid_vs_Lymphoid/Markers.Myeloid.bed",
    "Lymphoid":    f"{MARKERS_BASE}/coarse_Lymphoid_vs_Myeloid/Markers.Lymphoid.bed",
    "T_Cell":      f"{MARKERS_BASE}/one_vs_all/T_Cell_vs_all/Markers.T_Cell.bed",
    "B_Cell":      f"{MARKERS_BASE}/one_vs_all/B_Cell_vs_all/Markers.B_Cell.bed",
    "NK_Cell":     f"{MARKERS_BASE}/one_vs_all/NK_Cell_vs_all/Markers.NK_Cell.bed",
    "Monocyte":    f"{MARKERS_BASE}/one_vs_all/Monocyte_vs_all/Markers.Monocyte.bed",
    "Granulocyte": f"{MARKERS_BASE}/one_vs_all/Granulocyte_vs_all/Markers.Granulocyte.bed",
}

# ── Input PAT folders (all preprocessing batches) ────────────────
IN_PAT_PREFIXES = [
    f"{WORKSPACE_BUCKET}/results/bi_sequel",
    f"{WORKSPACE_BUCKET}/results/bi_revio",
    f"{WORKSPACE_BUCKET}/results/bcm_revio",
    f"{WORKSPACE_BUCKET}/results/bcm_sequel",
    f"{WORKSPACE_BUCKET}/results/bcm_ont",
    f"{WORKSPACE_BUCKET}/results/jhu_ont",
    f"{WORKSPACE_BUCKET}/results/uw_revio",
    f"{WORKSPACE_BUCKET}/results/uw_sequel",
    f"{WORKSPACE_BUCKET}/results/ha_revio",
    f"{WORKSPACE_BUCKET}/results/v7_base",
]

# ── Single-batch override (set to None to run all) ────────────────
ACTIVE_BATCH = None
if ACTIVE_BATCH is not None:
    IN_PAT_PREFIXES = [ACTIVE_BATCH]

print(f"MODE            : {MODE}")
print(f"TAU             : {TAU}  (log(tau)={math.log(TAU):.4f})")
print(f"MIN_HITS        : {MIN_HITS}")
print(f"OUT_ROOT_GCS    : {OUT_ROOT_GCS}")
print(f"Targets         : {list(TARGET_BEDS.keys())}")
print(f"Input prefixes  : {len(IN_PAT_PREFIXES)} folder(s)")
for p in IN_PAT_PREFIXES:
    print(f"  {p}")

## Cell 3 — Utility Functions

In [ ]:
def safe_label(s: str) -> str:
    "Convert arbitrary string to a filesystem-safe identifier."
    return re.sub(r"[^A-Za-z0-9_]+", "_", s)


def gsutil_ls(prefix: str) -> list:
    "List GCS objects matching prefix. Returns empty list on error."
    try:
        out = subprocess.check_output(
            ["gsutil", "ls", prefix], text=True, stderr=subprocess.DEVNULL
        )
        return [x.strip() for x in out.splitlines() if x.strip()]
    except subprocess.CalledProcessError:
        return []


def gcs_exists(path: str) -> bool:
    "Return True if a GCS object exists."
    try:
        subprocess.check_output(
            ["gsutil", "-q", "stat", path], text=True, stderr=subprocess.DEVNULL
        )
        return True
    except subprocess.CalledProcessError:
        return False


def list_all_pat_files(prefixes: list) -> list:
    "Collect all .pat.gz files across multiple GCS prefixes."
    pats = []
    for prefix in prefixes:
        found = gsutil_ls(f"{prefix}/*.pat.gz")
        pats.extend(found)
        print(f"  {prefix}: {len(found)} PAT files")
    return sorted(set(pats))


def make_sample_key(pat_gcs: str) -> str:
    "Derive a unique sample key from GCS PAT path: {batch_folder}__{sample_id}."
    p         = Path(pat_gcs)
    sample_id = p.name.replace(".pat.gz", "")
    return f"{p.parent.name}__{sample_id}"


def cleanup_files(paths: list):
    "Silently remove local files or directories."
    for p in paths:
        if p is None:
            continue
        p = Path(p)
        try:
            if p.is_file():  p.unlink()
            elif p.is_dir(): shutil.rmtree(p)
        except Exception as e:
            print(f"  Warning: cleanup failed for {p}: {e}")


def quantile_from_hist(hist: Counter, q: float) -> float:
    "Approximate q-th quantile from a Counter histogram of integer values."
    if not hist:
        return 0.0
    items = sorted(hist.items())
    total = sum(c for _, c in items)
    if total == 0:
        return 0.0
    thresh = q * (total - 1)
    cum = 0
    for val, cnt in items:
        if cum + cnt > thresh:
            return float(val)
        cum += cnt
    return float(items[-1][0])


print("Utility functions defined.")

## Cell 4 — Validate wgbstools and Load CpG Chromosome Sizes

`CpG.chrome.size` maps each chromosome to its CpG count and cumulative offset in the
genome-wide global CpG index — required to convert PAT indices to local array positions.

In [ ]:
assert TOOL_BIN.exists(),        f"wgbstools not found: {TOOL_BIN}"
assert CPG_CHROME_SIZE.exists(), f"CpG.chrome.size not found: {CPG_CHROME_SIZE}"
assert shutil.which("bgzip"),    "bgzip not found — install: sudo apt-get install tabix"
print("wgbstools OK:", TOOL_BIN)
print("bgzip      OK")


def load_cpg_chrom_sizes(path: Path):
    'Parse CpG.chrome.size -> (cs DataFrame, chrom_counts dict, chrom_offsets dict).'
    cs = pd.read_csv(path, sep="\t", header=None, names=["chrom", "n_cpg"])
    cs["n_cpg"]   = cs["n_cpg"].astype(np.int64)
    cs["offset"]  = cs["n_cpg"].cumsum().shift(fill_value=0).astype(np.int64)
    chrom_counts  = dict(zip(cs["chrom"], cs["n_cpg"]))
    chrom_offsets = dict(zip(cs["chrom"], cs["offset"]))
    return cs, chrom_counts, chrom_offsets


cs, chrom_counts, chrom_offsets = load_cpg_chrom_sizes(CPG_CHROME_SIZE)
print(f"Total CpGs in hg38: {int(cs['n_cpg'].sum()):,}")
display(cs.head(5))

## Cell 5 — CpG Index Mapping Helpers

Marker BEDs and PAT files may use either local (per-chromosome) or global (genome-wide)
CpG indices. These functions disambiguate both cases transparently.

In [ ]:
def marker_interval_to_local_slice(chrom, cpg_start_1based, cpg_end_1based,
                                   chrom_counts, chrom_offsets):
    'Convert a marker BED interval to (j0, j1) 0-based slice; None if invalid.'
    if chrom not in chrom_counts:
        return None
    n   = int(chrom_counts[chrom])
    off = int(chrom_offsets[chrom])
    if 1 <= cpg_start_1based <= n and 1 <= cpg_end_1based <= n + 1:
        local_start, local_end = cpg_start_1based, cpg_end_1based
    elif off + 1 <= cpg_start_1based <= off + n and off + 1 <= cpg_end_1based <= off + n + 1:
        local_start = cpg_start_1based - off
        local_end   = cpg_end_1based   - off
    else:
        return None
    j0 = max(0, min(int(local_start - 1), n))
    j1 = max(0, min(int(local_end   - 1), n))
    if j1 <= j0:
        return None
    return j0, j1


def pat_start_to_local_j(chrom, start_cpg_1based, chrom_counts, chrom_offsets, base=1):
    'Convert PAT start CpG index (1-based, local or global) to 0-based local j.'
    if chrom not in chrom_counts:
        return None
    n   = int(chrom_counts[chrom])
    off = int(chrom_offsets[chrom])
    if 1 <= start_cpg_1based <= n:
        local_1based = start_cpg_1based
    elif off + 1 <= start_cpg_1based <= off + n:
        local_1based = start_cpg_1based - off
    else:
        return None
    j = int(local_1based - base)
    if j < 0 or j >= n:
        return None
    return j


print("Index mapping helpers defined.")

## Cell 6 — Build Per-Chromosome Probability Atlas from a Marker BED

Reads a wgbstools marker BED file (columns 0,3,4,9,10) and builds dense float32 arrays
of target (`p_tg`) and background (`p_bg`) methylation probabilities indexed by local CpG position.

In [ ]:
def build_chrom_params_target_bg(markers_bed, chrom_counts, chrom_offsets, eps=1e-4):
    '
    Parse marker BED and build per-chromosome probability arrays.

    BED columns: 0=chrom, 3=cpg_start, 4=cpg_end, 9=p_tg, 10=p_bg.
    Returns (chrom_params dict, n_marker_sites int, df_shape tuple).
    '
    df = pd.read_csv(markers_bed, sep="\t", comment="#", header=None)
    if df.shape[1] < 11:
        raise ValueError(f"Marker BED has only {df.shape[1]} columns; need >= 11")

    use = df[[0, 3, 4, 9, 10]].copy()
    use.columns = ["chrom", "cpg_start", "cpg_end", "p_tg", "p_bg"]
    use["cpg_start"] = use["cpg_start"].astype(np.int64)
    use["cpg_end"]   = use["cpg_end"].astype(np.int64)
    use["p_tg"]      = use["p_tg"].astype(np.float32)
    use["p_bg"]      = use["p_bg"].astype(np.float32)

    chrom_params = {
        chrom: {
            "p_tg": np.full(int(n), np.nan, dtype=np.float32),
            "p_bg": np.full(int(n), np.nan, dtype=np.float32),
        }
        for chrom, n in chrom_counts.items()
    }
    for chrom, g in use.groupby("chrom"):
        if chrom not in chrom_params:
            continue
        p_tg_arr = chrom_params[chrom]["p_tg"]
        p_bg_arr = chrom_params[chrom]["p_bg"]
        for cpg_s, cpg_e, ptg, pbg in zip(g["cpg_start"].values, g["cpg_end"].values,
                                           g["p_tg"].values, g["p_bg"].values):
            sl = marker_interval_to_local_slice(
                chrom, int(cpg_s), int(cpg_e), chrom_counts, chrom_offsets)
            if sl is None:
                continue
            j0, j1 = sl
            p_tg_arr[j0:j1] = float(np.clip(ptg, eps, 1.0 - eps))
            p_bg_arr[j0:j1] = float(np.clip(pbg, eps, 1.0 - eps))

    total_sites = sum(int(np.isfinite(chrom_params[c]["p_tg"]).sum()) for c in chrom_params)
    return chrom_params, total_sites, df.shape


print("build_chrom_params_target_bg defined.")

## Cell 7 — Download All 7 Marker BEDs and Build Atlases

BED files are cached locally in `markers_all7/`. Atlas construction takes ~10 s per target.

In [ ]:
chrom_params_by_target = {}

for target, gcs_bed in TARGET_BEDS.items():
    local_bed = MARKERS_LOCAL / f"{safe_label(target)}.bed"
    if not local_bed.exists():
        print(f"Downloading {target} marker BED from GCS...")
        subprocess.run(["gsutil", "-m", "-q", "cp", gcs_bed, str(local_bed)], check=True)

    params, n_sites, shape = build_chrom_params_target_bg(local_bed, chrom_counts, chrom_offsets)
    chrom_params_by_target[target] = params
    print(f"  {target:12s}: {n_sites:8,} marker CpG sites  (BED rows={shape[0]:,})")

print(f"\nAll {len(chrom_params_by_target)} atlases ready.")

## Cell 8 — PAT Parsing Helpers

PAT format (tab-separated): `chrom  start_cpg_1based  pattern  multiplicity`

where `pattern` is a string of `C`/`T` characters (one per CpG in the fragment).

In [ ]:
PAT_LINE_RE  = re.compile(r"^(\S+)\t(\d+)\t(\S+)\t(\d+)\s*$")
METH_CHARS   = frozenset(["C", "1"])
UNMETH_CHARS = frozenset(["T", "0"])


def parse_pat_line(line: str):
    "Parse one PAT line. Returns (chrom, start_cpg, pattern, multiplicity) or None."
    m = PAT_LINE_RE.match(line)
    if not m:
        return None
    return m.group(1), int(m.group(2)), m.group(3), int(m.group(4))


def infer_pat_base(pat_gz: Path, n_lines: int = 20_000) -> int:
    "Inspect CpG start indices to detect local vs global coordinates (both 1-based)."
    mn = math.inf
    with gzip.open(pat_gz, "rt") as f:
        for i, line in enumerate(f):
            if i >= n_lines:
                break
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            try:
                mn = min(mn, int(parts[1]))
            except ValueError:
                pass
    return 1 if mn >= 1 else 0


print("PAT parsing helpers defined.")

## Cell 9 — LLR Scoring Function

For each read, the log-likelihood ratio against a target is:

$$\text{LLR} = \sum_{i \in \text{marker CpGs}} \log \frac{P(\text{obs}_i \mid \text{target})}{P(\text{obs}_i \mid \text{background})}$$

Only positions with finite (non-NaN) marker probabilities contribute.

In [ ]:
def llr_for_row_sparse(pattern, p_tg_seg, p_bg_seg, eps=1e-4):
    '
    Compute LLR for one PAT read against a target/background marker segment.
    Returns (llr float, hits int).
    '
    mask = np.isfinite(p_tg_seg) & np.isfinite(p_bg_seg)
    if not mask.any():
        return 0.0, 0
    idxs  = np.flatnonzero(mask)
    pt    = np.clip(p_tg_seg[idxs], eps, 1.0 - eps)
    pb    = np.clip(p_bg_seg[idxs], eps, 1.0 - eps)
    log_m = np.log(pt)    - np.log(pb)
    log_u = np.log1p(-pt) - np.log1p(-pb)
    llr  = 0.0
    hits = 0
    for k, i in enumerate(idxs):
        ch = pattern[i]
        if ch in METH_CHARS:
            llr  += float(log_m[k]);  hits += 1
        elif ch in UNMETH_CHARS:
            llr  += float(log_u[k]);  hits += 1
    return llr, hits


print("LLR scoring function defined.")

## Cell 10 — bgzip / wgbstools index / pat2beta Helpers

In [ ]:
def bgzip_file(in_txt: Path) -> Path:
    "bgzip a plain-text PAT file in place. Returns path to the .gz output."
    out_gz = Path(str(in_txt) + ".gz")
    if out_gz.exists(): out_gz.unlink()
    subprocess.run(["bgzip", "-f", str(in_txt)], check=True)
    assert out_gz.exists(), f"bgzip failed: {out_gz}"
    return out_gz


def wgbstools_index(pat_gz: Path) -> Path:
    "Create a .csi tabix index for a bgzipped PAT file."
    csi = Path(str(pat_gz) + ".csi")
    if csi.exists(): csi.unlink()
    subprocess.run([str(TOOL_BIN), "index", str(pat_gz)], check=True)
    assert csi.exists(), f"wgbstools index failed: {csi}"
    return csi


def pat2beta(pat_gz, out_dir, genome="hg38", threads=4, force=True):
    "Run wgbstools pat2beta. Returns path to .beta output."
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [str(TOOL_BIN), "pat2beta"]
    if force:
        cmd.append("-f")
    cmd += ["--genome", genome, "--threads", str(threads),
            "--out_dir", str(out_dir), str(pat_gz)]
    subprocess.run(cmd, check=True)
    stem = pat_gz.name
    if stem.endswith(".pat.gz"): stem = stem[:-7]
    cand = out_dir / f"{stem}.beta"
    if cand.exists(): return cand
    betas = sorted(out_dir.glob("*.beta"), key=lambda p: p.stat().st_mtime, reverse=True)
    if betas: return betas[0]
    raise FileNotFoundError(f"pat2beta produced no .beta in {out_dir}")


print("bgzip / index / pat2beta helpers defined.")

## Cell 11 — Process One Sample (All 7 Targets in a Single PAT Pass)

Downloads PAT, streams every read scoring it against all 7 atlases simultaneously,
writes passing reads to per-target PAT files, then bgzip + index + pat2beta + upload.

In [ ]:
def process_one_sample(pat_gcs, out_root_gcs, min_hits=5, tau=1.053,
                       threads=4, make_beta=True, skip_if_exists=True):
    'Sort one bulk PAT file into 7 cell-type-specific PAT (+ beta) files.'
    sample_id  = Path(pat_gcs).name.replace(".pat.gz", "")
    sample_key = make_sample_key(pat_gcs)
    tau_log    = math.log(tau)
    print(f"\n=== {sample_key} ===")

    if skip_if_exists:
        probe = f"{out_root_gcs}/Myeloid/{sample_key}_Myeloid.pat.gz"
        if gcs_exists(probe):
            print(f"  Skip (exists): {probe}")
            return {"sample_id": sample_id, "sample_key": sample_key, "status": "skipped_exists"}

    local_in = TMP / f"{sample_key}.pat.gz"
    print(f"  Downloading {pat_gcs} ...")
    subprocess.run(["gsutil", "-m", "-q", "cp", pat_gcs, str(local_in)], check=True)
    base = infer_pat_base(local_in)

    out_txt = {}
    out_fh  = {}
    for target in TARGET_BEDS:
        p = TMP / f"{sample_key}_{safe_label(target)}.pat"
        if p.exists(): p.unlink()
        out_txt[target] = p
        out_fh[target]  = open(p, "wt")

    counts    = {t: {"Target": 0, "Other": 0, "AmbiguousTie": 0,
                     "LowHits": 0, "NoOverlap": 0, "Skipped": 0}
                 for t in TARGET_BEDS}
    hits_hist = {t: Counter() for t in TARGET_BEDS}

    with gzip.open(local_in, "rt") as fin:
        for line in fin:
            parsed = parse_pat_line(line)
            if parsed is None:
                for t in TARGET_BEDS: counts[t]["Skipped"] += 1
                continue
            chrom, start_cpg, pattern, n_reads = parsed
            if chrom not in chrom_counts:
                for t in TARGET_BEDS: counts[t]["Skipped"] += n_reads
                continue
            j = pat_start_to_local_j(chrom, start_cpg, chrom_counts, chrom_offsets, base=base)
            if j is None:
                for t in TARGET_BEDS: counts[t]["Skipped"] += n_reads
                continue
            L = len(pattern)
            if j + L > int(chrom_counts[chrom]):
                for t in TARGET_BEDS: counts[t]["Skipped"] += n_reads
                continue
            for target, params in chrom_params_by_target.items():
                p_tg_seg = params[chrom]["p_tg"][j : j + L]
                p_bg_seg = params[chrom]["p_bg"][j : j + L]
                llr, hits = llr_for_row_sparse(pattern, p_tg_seg, p_bg_seg)
                hits_hist[target][hits] += 1
                if hits == 0:
                    counts[target]["NoOverlap"] += n_reads
                elif hits < min_hits:
                    counts[target]["LowHits"] += n_reads
                elif llr > tau_log:
                    counts[target]["Target"] += n_reads
                    out_fh[target].write(line)
                elif llr < -tau_log:
                    counts[target]["Other"] += n_reads
                else:
                    counts[target]["AmbiguousTie"] += n_reads

    for fh in out_fh.values():
        fh.close()

    produced_targets = []
    for target in TARGET_BEDS:
        txt = out_txt[target]
        if not txt.exists() or txt.stat().st_size == 0:
            if txt.exists(): txt.unlink()
            continue
        gz  = bgzip_file(txt)
        csi = wgbstools_index(gz)
        produced = [gz, csi]
        if make_beta:
            beta_dir = TMP / f"{sample_key}_{safe_label(target)}_beta"
            beta     = pat2beta(gz, beta_dir, genome="hg38", threads=threads, force=True)
            produced.append(beta)
        dst = f"{out_root_gcs}/{safe_label(target)}/"
        subprocess.run(["gsutil", "-m", "-q", "cp", *[str(p) for p in produced], dst], check=True)
        print(f"  {target}: {counts[target]['Target']:>8,} target reads -> {dst}")
        cleanup_files([gz, csi])
        if make_beta: cleanup_files([beta_dir])
        produced_targets.append(target)

    cleanup_files([local_in])

    row = {"sample_id": sample_id, "sample_key": sample_key, "status": "ok",
           "min_hits": min_hits, "tau": tau, "made_targets": ",".join(produced_targets)}
    for target in TARGET_BEDS:
        h = hits_hist[target]
        row[f"{target}_median_hits"] = quantile_from_hist(h, 0.5)
        row[f"{target}_p90_hits"]    = quantile_from_hist(h, 0.9)
        for k, v in counts[target].items():
            row[f"{target}_{k}"] = int(v)
    return row


print("process_one_sample defined.")

## Cell 12 — Batch Runner

Discovers all PAT files across the configured input prefixes and runs
`process_one_sample` sequentially. For parallel execution use **04b_dsub_sorting.ipynb**.

In [ ]:
def run_batch(in_prefixes, out_root_gcs, max_samples=None, min_hits=5,
              tau=1.053, threads=4, make_beta=True, skip_if_exists=True):
    "Sort all PAT files found under in_prefixes into per-cell-type outputs."
    print("Discovering PAT files...")
    pats = list_all_pat_files(in_prefixes)
    print(f"Total: {len(pats)} PAT files across {len(in_prefixes)} prefix(es)")
    if max_samples is not None:
        pats = pats[:max_samples]
        print(f"Capped at {max_samples} samples.")

    rows = []
    for i, pat_gcs in enumerate(pats):
        print(f"\n[{i+1}/{len(pats)}] {Path(pat_gcs).name}")
        rows.append(process_one_sample(
            pat_gcs=pat_gcs, out_root_gcs=out_root_gcs,
            min_hits=min_hits, tau=tau, threads=threads,
            make_beta=make_beta, skip_if_exists=skip_if_exists,
        ))

    stats      = pd.DataFrame(rows)
    stats_path = TMP / "sorting_stats.csv"
    stats.to_csv(stats_path, index=False)
    subprocess.run(
        ["gsutil", "-m", "-q", "cp", str(stats_path), f"{out_root_gcs}/sorting_stats.csv"],
        check=True
    )
    print(f"\nStats saved to {out_root_gcs}/sorting_stats.csv")
    return stats


print("run_batch defined.")

## Cell 13 — TEST Run (3 Samples)

Processes `TEST_SAMPLE_LIMIT` samples with `skip_if_exists=False`.
Set `MODE = "TEST"` in Cell 2.

In [ ]:
if MODE == "TEST":
    print(f"=== TEST RUN: {TEST_SAMPLE_LIMIT} samples ===")
    stats_test = run_batch(
        in_prefixes=IN_PAT_PREFIXES, out_root_gcs=OUT_ROOT_GCS,
        max_samples=TEST_SAMPLE_LIMIT, min_hits=MIN_HITS, tau=TAU,
        threads=THREADS, make_beta=MAKE_BETA, skip_if_exists=False,
    )
    display(stats_test)
else:
    print("MODE='PRODUCTION' — skipping TEST cell.")

## Cell 14 — PRODUCTION Run (All Batches)

Processes all samples; already-sorted samples are skipped.
Set `MODE = "PRODUCTION"` in Cell 2.
Runtime ~25 min/sample; for large cohorts use 04b_dsub_sorting.ipynb.

In [ ]:
if MODE == "PRODUCTION":
    print("=== PRODUCTION RUN — all batches ===")
    stats_prod = run_batch(
        in_prefixes=IN_PAT_PREFIXES, out_root_gcs=OUT_ROOT_GCS,
        max_samples=None, min_hits=MIN_HITS, tau=TAU,
        threads=THREADS, make_beta=MAKE_BETA, skip_if_exists=True,
    )
    display(stats_prod.head(10))
    print(f"\nTotal samples processed: {len(stats_prod)}")
    print(stats_prod["status"].value_counts().to_string())
else:
    print("MODE='TEST' — skipping PRODUCTION cell.")

## Cell 15 — QC: Sorting Yield Summary

Plots distribution of per-target read assignment rates across all processed samples.

In [ ]:
stats_gcs   = f"{OUT_ROOT_GCS}/sorting_stats.csv"
local_stats = TMP / "sorting_stats_qc.csv"
try:
    subprocess.run(["gsutil", "-m", "-q", "cp", stats_gcs, str(local_stats)], check=True)
    stats = pd.read_csv(local_stats)
    print(f"Loaded {len(stats)} rows from {stats_gcs}")
except subprocess.CalledProcessError:
    print(f"Stats not yet available at {stats_gcs}. Run Cell 13 or 14 first.")
    stats = None

if stats is not None and len(stats) > 0:
    ok      = stats[stats["status"] == "ok"].copy()
    targets = list(TARGET_BEDS.keys())

    for t in targets:
        dcols = [f"{t}_{k}" for k in ["Target", "Other", "AmbiguousTie", "LowHits"]
                 if f"{t}_{k}" in ok.columns]
        if dcols:
            ok[f"{t}_yield"] = ok[f"{t}_Target"] / ok[dcols].sum(axis=1).clip(lower=1)

    yield_cols = [f"{t}_yield" for t in targets if f"{t}_yield" in ok.columns]
    if yield_cols:
        fig, axes = plt.subplots(1, len(yield_cols), figsize=(3*len(yield_cols), 4), sharey=True)
        if len(yield_cols) == 1: axes = [axes]
        for ax, col in zip(axes, yield_cols):
            vals = ok[col].dropna() * 100
            ax.hist(vals, bins=20, color="steelblue", edgecolor="white", lw=0.4)
            ax.axvline(vals.median(), color="crimson", ls="--", lw=1.2,
                       label=f"median {vals.median():.1f}%")
            ax.set_title(col.replace("_yield",""), fontsize=9)
            ax.set_xlabel("% Assigned", fontsize=8)
            ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.0f%%"))
            ax.legend(fontsize=7)
        axes[0].set_ylabel("Samples", fontsize=8)
        plt.suptitle("Cell-Type Sorting Yield", fontsize=11, y=1.02)
        plt.tight_layout(); plt.show()

        summary = (ok[yield_cols].describe().T * 100).round(2)
        summary.index = [c.replace("_yield","") for c in summary.index]
        print("\nYield summary (% reads as Target):")
        display(summary[["mean","50%","std","min","max"]])